This notebook creates the reusable gold-layer monthly BI view `adwm_wh.gold.vw_bi_monthly_sales_trend` for downstream trend reporting.

Scope:

* Create or replace the monthly trend BI view in `adwm_wh.gold`
* Reuse `adwm_wh.gold.vw_bi_factsales_base` as the source layer
* Provide a time-series-friendly month grain
* Run lightweight validation for row coverage and top monthly results

Business grain:

* One row per year-month

In [0]:
%sql
CREATE OR REPLACE VIEW adwm_wh.gold.vw_bi_monthly_sales_trend AS
SELECT
  YearNumber,
  MonthNumber,
  MonthName,
  CONCAT(CAST(YearNumber AS STRING), '-', LPAD(CAST(MonthNumber AS STRING), 2, '0')) AS YearMonth,
  MIN(FullDate) AS PeriodStartDate,
  MAX(FullDate) AS PeriodEndDate,
  COUNT(*) AS SalesLineCount,
  COUNT(DISTINCT SalesOrderNumber) AS SalesOrderCount,
  SUM(OrderQuantity) AS TotalOrderQuantity,
  CAST(SUM(SalesAmount) AS DECIMAL(19,4)) AS TotalSalesAmount,
  CAST(SUM(TotalCost) AS DECIMAL(19,4)) AS TotalCost,
  CAST(SUM(DiscountAmount) AS DECIMAL(19,4)) AS TotalDiscountAmount,
  CAST(SUM(GrossMargin) AS DECIMAL(19,4)) AS GrossMargin
FROM adwm_wh.gold.vw_bi_factsales_base
GROUP BY YearNumber, MonthNumber, MonthName;

In [0]:
%sql
SELECT
  YearMonth,
  PeriodStartDate,
  PeriodEndDate,
  SalesLineCount,
  SalesOrderCount,
  TotalOrderQuantity,
  CAST(TotalSalesAmount AS DECIMAL(19,2)) AS TotalSalesAmount,
  CAST(GrossMargin AS DECIMAL(19,2)) AS GrossMargin,
  CAST(TotalDiscountAmount AS DECIMAL(19,2)) AS TotalDiscountAmount
FROM adwm_wh.gold.vw_bi_monthly_sales_trend
ORDER BY YearMonth DESC
LIMIT 12;